In [1]:
import os

In [3]:
"""
AI Activity Assistant — Dark UI  |  Gradio 6.7 compatible
Features:
  • Send button to the right of the query textbox
  • Streaming: answer typed out word-by-word
  • Pipeline inspector accordion
"""

import gradio as gr
import json
import time
from agents import run_query

# ─────────────────────────────────────────────────────────────────────────────
_last_pipeline: dict = {}


def chat_fn(message: str, history: list):
    """
    Generator function — Gradio streams each yield() to the UI live.
    Step 1: instant "thinking" placeholder
    Step 2: run full agent pipeline
    Step 3: stream final_answer word-by-word
    """
    global _last_pipeline

    # Immediate feedback
    yield "⏳ Agents are working on your request…"

    try:
        result = run_query(message)
    except Exception as e:
        _last_pipeline = {"error": str(e)}
        yield f"❌ Error: {e}"
        return

    _last_pipeline = {
        "intent_plan":       result.get("intent_plan"),
        "active_tasks":      result.get("active_tasks"),
        "weather_condition": result.get("weather_condition"),
        "weather":           result.get("weather_result"),
        "outdoor":           result.get("outdoor_result"),
        "entertainment":     result.get("entertainment_result"),
        "food":              result.get("food_result"),
        "price":             result.get("price_result"),
        "general":           result.get("general_result"),
    }

    # Stream word-by-word
    final_answer: str = result.get("final_answer", "No answer generated.")
    words = final_answer.split(" ")
    accumulated = ""
    for i, word in enumerate(words):
        accumulated += ("" if i == 0 else " ") + word
        yield accumulated
        time.sleep(0.02)   # adjust 0.01–0.05 for typing speed


def get_pipeline() -> str:
    return json.dumps(_last_pipeline, indent=2)


SAMPLES = [
    "Hiking today in Mumbai, movie if it rains",
    "Find a painting spot in Pune today",
    "Sushi + live concert tonight in Delhi",
    "Things to do this weekend under ₹500 in Bangalore",
]

CSS = """
@import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;500&family=Sora:wght@300;400;600&display=swap');

*, *::before, *::after { box-sizing: border-box; }

/* ── Global dark canvas ── */
body,
.gradio-container,
.main, .wrap, .contain, .gap,
.block, .gr-block, .gr-box, .gr-form, .gr-panel, .gr-group {
    background: #080808 !important;
    border: none !important;
    box-shadow: none !important;
    font-family: 'Sora', sans-serif !important;
    color: #d4d4d4 !important;
}
.gradio-container {
    max-width: 98% !important;
    margin: 0 auto !important;
    padding: 0 24px 40px !important;
}

/* ── Header ── */
#app-header {
    padding: 52px 0 32px;
    text-align: center;
    border-bottom: 1px solid #141414;
    margin-bottom: 28px;
}
#app-header h1 {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 1.55rem;
    font-weight: 500;
    color: #f0f0f0;
    letter-spacing: -0.5px;
}
#app-header h1 span { color: #3ecf8e; }
#app-header p {
    font-size: 0.78rem;
    color: #333;
    margin-top: 10px;
    font-family: 'IBM Plex Mono', monospace;
    letter-spacing: 0.4px;
}

/* ── Sample pills ── */
#samples {
    display: flex !important;
    flex-wrap: wrap !important;
    gap: 8px !important;
    margin-bottom: 20px !important;
}
#samples button {
    background: transparent !important;
    border: 1px solid #1e1e1e !important;
    color: #555 !important;
    border-radius: 20px !important;
    font-size: 0.73rem !important;
    padding: 5px 14px !important;
    font-family: 'IBM Plex Mono', monospace !important;
    cursor: pointer !important;
    transition: all .18s ease !important;
    white-space: nowrap !important;
}
#samples button:hover {
    border-color: #3ecf8e !important;
    color: #3ecf8e !important;
    background: rgba(62,207,142,0.05) !important;
}

/* ── Chatbot ── */
.chatbot, .chatbot > div {
    background: transparent !important;
    border: none !important;
}
.chatbot .placeholder {
    color: #1e1e1e !important;
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 0.82rem !important;
}

/* User bubble */
.message.user > div,
div[data-testid="user"] > div {
    background: #111 !important;
    border: 1px solid #1e1e1e !important;
    border-radius: 14px 14px 4px 14px !important;
    color: #e0e0e0 !important;
    font-size: 0.87rem !important;
    padding: 12px 16px !important;
    line-height: 1.6 !important;
}

/* Bot message */
.message.bot > div,
div[data-testid="bot"] > div {
    background: transparent !important;
    border: none !important;
    border-left: 2px solid #1e1e1e !important;
    border-radius: 0 !important;
    padding: 4px 0 4px 16px !important;
    color: #999 !important;
    font-size: 0.875rem !important;
    line-height: 1.75 !important;
}

/* ── Query textbox ── */
#query-box textarea {
    background: #0e0e0e !important;
    border: 1px solid #1e1e1e !important;
    border-radius: 12px !important;
    color: #e0e0e0 !important;
    font-family: 'Sora', sans-serif !important;
    font-size: 0.88rem !important;
    padding: 14px 16px !important;
    resize: none !important;
    transition: border-color .2s !important;
    min-height: 52px !important;
    line-height: 1.55 !important;
}
#query-box textarea:focus {
    border-color: #2a2a2a !important;
    outline: none !important;
    box-shadow: none !important;
}
#query-box textarea::placeholder { color: #2a2a2a !important; }

/* ── SEND BUTTON ──
   Gradio 6 renders submit_btn (string form) as <button class="primary">.
   We style it green and fixed-height to sit flush beside the textbox. */
button.primary,
.submit-btn button {
    background: #3ecf8e !important;
    color: #080808 !important;
    border: none !important;
    border-radius: 12px !important;
    font-family: 'IBM Plex Mono', monospace !important;
    font-weight: 500 !important;
    font-size: 0.82rem !important;
    padding: 0 24px !important;
    height: 52px !important;
    min-width: 96px !important;
    cursor: pointer !important;
    white-space: nowrap !important;
    letter-spacing: 0.3px !important;
    transition: background .18s ease, transform .1s ease !important;
    flex-shrink: 0 !important;
}
button.primary:hover,
.submit-btn button:hover {
    background: #2eb87a !important;
    transform: translateY(-1px) !important;
}
button.primary:active { transform: translateY(0) !important; }

/* ── Pipeline accordion ── */
#pipeline-toggle {
    margin-top: 24px;
    border-top: 1px solid #111 !important;
    padding-top: 16px;
}
#pipeline-toggle .label-wrap span {
    display: block !important;
    color: #2e2e2e !important;
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 0.73rem !important;
    letter-spacing: 0.5px !important;
}
#pipeline-toggle pre,
#pipeline-toggle code {
    background: #0a0a0a !important;
    border: 1px solid #141414 !important;
    border-radius: 10px !important;
    color: #3ecf8e !important;
    font-size: 0.73rem !important;
    font-family: 'IBM Plex Mono', monospace !important;
}

/* Refresh button */
button.secondary {
    background: transparent !important;
    border: 1px solid #1a1a1a !important;
    color: #3a3a3a !important;
    border-radius: 10px !important;
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 0.72rem !important;
    transition: all .15s !important;
}
button.secondary:hover {
    border-color: #333 !important;
    color: #777 !important;
}

/* Hide noisy Gradio chrome */
label > span, footer, .built-with { display: none !important; }

/* Scrollbar */
::-webkit-scrollbar { width: 4px; }
::-webkit-scrollbar-track { background: #080808; }
::-webkit-scrollbar-thumb { background: #1a1a1a; border-radius: 2px; }
"""

# ─────────────────────────────────────────────────────────────────────────────
with gr.Blocks(title="AI Activity Assistant") as demo:

    gr.HTML("""
    <div id="app-header">
      <h1>AI <span>Activity</span> Assistant</h1>
      <p>intent-driven · multi-agent · llama 3.3 · langgraph · tavily · openweathermap</p>
    </div>
    """)

    # Sample pills — outside ChatInterface so we can wire them to ci.textbox
    with gr.Row(elem_id="samples"):
        s_btns = [gr.Button(s, size="sm") for s in SAMPLES]

    # ── ChatInterface ─────────────────────────────────────────────────────
    # Gradio 6.7 rules (from official docs):
    #   • gr.Chatbot() takes: height, show_label, placeholder  (NO type param)
    #   • submit_btn    → plain string label, NOT a gr.Button object
    #   • streaming     → just make chat_fn a generator (yield instead of return)
    ci = gr.ChatInterface(
        fn=chat_fn,
        chatbot=gr.Chatbot(
            height=440,
            show_label=False,
            placeholder=(
                "<strong style='color:#1e1e1e;"
                "font-family:IBM Plex Mono,monospace;"
                "font-size:0.8rem'>"
                "ask me what you want to do today…</strong>"
            ),
        ),
        textbox=gr.Textbox(
            placeholder="e.g. Hiking in Mumbai today, movie if it rains…",
            container=False,
            scale=7,
            elem_id="query-box",
        ),
        submit_btn="Send →",
        examples=None,
        cache_examples=False,
    )

    # Wire sample pills → textbox
    for btn, sample in zip(s_btns, SAMPLES):
        btn.click(lambda s=sample: s, outputs=ci.textbox)

    # Pipeline inspector
    with gr.Accordion("⚙ pipeline inspector", open=False, elem_id="pipeline-toggle"):
        refresh_btn  = gr.Button("↺ refresh", variant="secondary", size="sm")
        pipeline_out = gr.Code(value="{}", language="json", lines=18, show_label=False)
        refresh_btn.click(fn=get_pipeline, outputs=pipeline_out)

    # Auto-refresh pipeline JSON after every message
    ci.textbox.submit(fn=get_pipeline, outputs=pipeline_out)


if __name__ == "__main__":
    demo.launch(
        server_name="0.0.0.0",
        server_port=7860,
        show_error=True,
        theme=gr.themes.Base(),
        css=CSS,
        share=True,
    )

C:\Users\Kundan\.conda\envs\env_jupyter\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [1]:
"""
AI Activity Assistant — Dark UI  |  Gradio 6.7 compatible with TRUE STREAMING
Features:
  • Send button to the right of the query textbox
  • TRUE streaming: answer updates as agents work
  • Pipeline inspector accordion
"""

import gradio as gr
import json
from agents import stream_query

# ─────────────────────────────────────────────────────────────────────────────
_last_pipeline: dict = {}


def chat_fn(message: str, history: list):
    """
    Generator function — Gradio streams each yield() to the UI live.
    Uses stream_query() from agents.py for REAL-TIME streaming.
    """
    global _last_pipeline

    try:
        # Use the stream_query generator from agents.py
        # This yields incremental updates as agents complete work
        for streamed_output in stream_query(message):
            yield streamed_output
            
    except Exception as e:
        _last_pipeline = {"error": str(e)}
        yield f"❌ Error: {e}"
        return


def get_pipeline() -> str:
    return json.dumps(_last_pipeline, indent=2)


SAMPLES = [
    "Hiking today in Mumbai, movie if it rains",
    "Find a painting spot in Pune today",
    "Sushi + live concert tonight in Delhi",
    "Things to do this weekend under ₹500 in Bangalore",
]

CSS = """
@import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;500&family=Sora:wght@300;400;600&display=swap');

*, *::before, *::after { box-sizing: border-box; }

/* ── Global dark canvas ── */
body,
.gradio-container,
.main, .wrap, .contain, .gap,
.block, .gr-block, .gr-box, .gr-form, .gr-panel, .gr-group {
    background: #080808 !important;
    border: none !important;
    box-shadow: none !important;
    font-family: 'Sora', sans-serif !important;
    color: #d4d4d4 !important;
}
.gradio-container {
    max-width: 98% !important;
    margin: 0 auto !important;
    padding: 0 24px 40px !important;
}

/* ── Header ── */
#app-header {
    padding: 52px 0 32px;
    text-align: center;
    border-bottom: 1px solid #141414;
    margin-bottom: 28px;
}
#app-header h1 {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 1.55rem;
    font-weight: 500;
    color: #f0f0f0;
    letter-spacing: -0.5px;
}
#app-header h1 span { color: #3ecf8e; }
#app-header p {
    font-size: 0.78rem;
    color: #333;
    margin-top: 10px;
    font-family: 'IBM Plex Mono', monospace;
    letter-spacing: 0.4px;
}

/* ── Sample pills ── */
#samples {
    display: flex !important;
    flex-wrap: wrap !important;
    gap: 8px !important;
    margin-bottom: 20px !important;
}
#samples button {
    background: transparent !important;
    border: 1px solid #1e1e1e !important;
    color: #555 !important;
    border-radius: 20px !important;
    font-size: 0.73rem !important;
    padding: 5px 14px !important;
    font-family: 'IBM Plex Mono', monospace !important;
    cursor: pointer !important;
    transition: all .18s ease !important;
    white-space: nowrap !important;
}
#samples button:hover {
    border-color: #3ecf8e !important;
    color: #3ecf8e !important;
    background: rgba(62,207,142,0.05) !important;
}

/* ── Chatbot ── */
.chatbot, .chatbot > div {
    background: transparent !important;
    border: none !important;
}
.chatbot .placeholder {
    color: #1e1e1e !important;
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 0.82rem !important;
}

/* User bubble */
.message.user > div,
div[data-testid="user"] > div {
    background: #111 !important;
    border: 1px solid #1e1e1e !important;
    border-radius: 14px 14px 4px 14px !important;
    color: #e0e0e0 !important;
    font-size: 0.87rem !important;
    padding: 12px 16px !important;
    line-height: 1.6 !important;
}

/* Bot message */
.message.bot > div,
div[data-testid="bot"] > div {
    background: transparent !important;
    border: none !important;
    border-left: 2px solid #1e1e1e !important;
    border-radius: 0 !important;
    padding: 4px 0 4px 16px !important;
    color: #999 !important;
    font-size: 0.875rem !important;
    line-height: 1.75 !important;
}

/* ── Query textbox ── */
#query-box textarea {
    background: #0e0e0e !important;
    border: 1px solid #1e1e1e !important;
    border-radius: 12px !important;
    color: #e0e0e0 !important;
    font-family: 'Sora', sans-serif !important;
    font-size: 0.88rem !important;
    padding: 14px 16px !important;
    resize: none !important;
    transition: border-color .2s !important;
    min-height: 52px !important;
    line-height: 1.55 !important;
}
#query-box textarea:focus {
    border-color: #2a2a2a !important;
    outline: none !important;
    box-shadow: none !important;
}
#query-box textarea::placeholder { color: #2a2a2a !important; }

/* ── SEND BUTTON ──
   Gradio 6 renders submit_btn (string form) as <button class="primary">.
   We style it green and fixed-height to sit flush beside the textbox. */
button.primary,
.submit-btn button {
    background: #3ecf8e !important;
    color: #080808 !important;
    border: none !important;
    border-radius: 12px !important;
    font-family: 'IBM Plex Mono', monospace !important;
    font-weight: 500 !important;
    font-size: 0.82rem !important;
    padding: 0 24px !important;
    height: 52px !important;
    min-width: 96px !important;
    cursor: pointer !important;
    white-space: nowrap !important;
    letter-spacing: 0.3px !important;
    transition: background .18s ease, transform .1s ease !important;
    flex-shrink: 0 !important;
}
button.primary:hover,
.submit-btn button:hover {
    background: #2eb87a !important;
    transform: translateY(-1px) !important;
}
button.primary:active { transform: translateY(0) !important; }

/* ── Pipeline accordion ── */
#pipeline-toggle {
    margin-top: 24px;
    border-top: 1px solid #111 !important;
    padding-top: 16px;
}
#pipeline-toggle .label-wrap span {
    display: block !important;
    color: #2e2e2e !important;
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 0.73rem !important;
    letter-spacing: 0.5px !important;
}
#pipeline-toggle pre,
#pipeline-toggle code {
    background: #0a0a0a !important;
    border: 1px solid #141414 !important;
    border-radius: 10px !important;
    color: #3ecf8e !important;
    font-size: 0.73rem !important;
    font-family: 'IBM Plex Mono', monospace !important;
}

/* Refresh button */
button.secondary {
    background: transparent !important;
    border: 1px solid #1a1a1a !important;
    color: #3a3a3a !important;
    border-radius: 10px !important;
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 0.72rem !important;
    transition: all .15s !important;
}
button.secondary:hover {
    border-color: #333 !important;
    color: #777 !important;
}

/* Hide noisy Gradio chrome */
label > span, footer, .built-with { display: none !important; }

/* Scrollbar */
::-webkit-scrollbar { width: 4px; }
::-webkit-scrollbar-track { background: #080808; }
::-webkit-scrollbar-thumb { background: #1a1a1a; border-radius: 2px; }
"""

# ─────────────────────────────────────────────────────────────────────────────
with gr.Blocks(title="AI Activity Assistant", css=CSS) as demo:

    gr.HTML("""
    <div id="app-header">
      <h1>AI <span>Activity</span> Assistant</h1>
      <p>intent-driven · multi-agent · llama 3.3 · langgraph · tavily · openweathermap</p>
    </div>
    """)

    # Sample pills — outside ChatInterface so we can wire them to ci.textbox
    with gr.Row(elem_id="samples"):
        s_btns = [gr.Button(s, size="sm") for s in SAMPLES]

    # ── ChatInterface ─────────────────────────────────────────────────────
    # Gradio 6.7 rules (from official docs):
    #   • gr.Chatbot() takes: height, show_label, placeholder  (NO type param)
    #   • submit_btn    → plain string label, NOT a gr.Button object
    #   • streaming     → just make chat_fn a generator (yield instead of return)
    ci = gr.ChatInterface(
        fn=chat_fn,
        chatbot=gr.Chatbot(
            height=440,
            show_label=False,
            placeholder=(
                "<strong style='color:#1e1e1e;"
                "font-family:IBM Plex Mono,monospace;"
                "font-size:0.8rem'>"
                "ask me what you want to do today…</strong>"
            ),
        ),
        textbox=gr.Textbox(
            placeholder="e.g. Hiking in Mumbai today, movie if it rains…",
            container=False,
            scale=7,
            elem_id="query-box",
        ),
        submit_btn="Send →",
        examples=None,
        cache_examples=False,
    )

    # Wire sample pills → textbox
    for btn, sample in zip(s_btns, SAMPLES):
        btn.click(lambda s=sample: s, outputs=ci.textbox)

    # Pipeline inspector
    with gr.Accordion("⚙ pipeline inspector", open=False, elem_id="pipeline-toggle"):
        refresh_btn  = gr.Button("↺ refresh", variant="secondary", size="sm")
        pipeline_out = gr.Code(value="{}", language="json", lines=18, show_label=False)
        refresh_btn.click(fn=get_pipeline, outputs=pipeline_out)

    # Auto-refresh pipeline JSON after every message
    ci.textbox.submit(fn=get_pipeline, outputs=pipeline_out)


if __name__ == "__main__":
    demo.launch(
        server_name="0.0.0.0",
        server_port=7860,
        show_error=True,
        theme=gr.themes.Base(),
        share=True,
    )

C:\Users\Kundan\.conda\envs\env_jupyter\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Kundan\AppData\Local\Temp\ipykernel_8024\1917705511.py:250: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="AI Activity Assistant", css=CSS) as demo:
C:\Users\Kundan\AppData\Local\Temp\ipykernel_8024\1917705511.py:268: UserWarning: You provided a custom `textbox` component, but also specified `submit_btn` parameter(s) on `gr.ChatInterface`. These ChatInterface parameters will be ignored. To customize these settings, pass them directly to your `gr.Textbox` or `gr.MultimodalTextbox` component instead. For example: textbox=gr.Textbox(..., submit_btn='submit')
  ci = gr.ChatInterface(

* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://733fc518a25200ad54.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
